In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

I0000 00:00:1775286993.316592     739 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775286996.557175     739 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775287033.548204     739 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[]


W0000 00:00:1775287052.677860     739 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amineipad/drone-sound-audio-detection")

print("Path to dataset files:", path)

ModuleNotFoundError: No module named 'kagglehub'

In [ ]:
import os

# Assuming 'path' variable from the previous cell holds the dataset directory
if 'path' in locals():
    print("Listing contents of the downloaded dataset directory:")
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')
else:
    print("The 'path' variable is not defined. Please ensure the Kaggle dataset download cell has been executed.")

The plots above show:

*   **Model Accuracy**: How well the model performed on the training and validation datasets over each epoch. A rising curve indicates improving accuracy.
*   **Model Loss**: The error rate of the model on the training and validation datasets over each epoch. A decreasing curve indicates the model is learning and reducing its errors.

Observing these plots can help identify if the model is overfitting (training accuracy much higher than validation accuracy) or underfitting (both accuracies are low).

In [ ]:
import os
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

# Define paths for drone and non-drone sounds
drone_path = os.path.join(path, 'Binary_Drone_Audio', 'yes_drone')
no_drone_path = os.path.join(path, 'Binary_Drone_Audio', 'unknown') # Corrected path
# silence_path = os.path.join(path, 'Binary_Drone_Audio', 'silence') # Removed as it doesn't exist

# Get a list of all audio files
drone_files = [os.path.join(drone_path, f) for f in os.listdir(drone_path) if f.endswith('.wav')]
no_drone_files = [os.path.join(no_drone_path, f) for f in os.listdir(no_drone_path) if f.endswith('.wav')]
# silence_files = [os.path.join(silence_path, f) for f in os.listdir(silence_path) if f.endswith('.wav')] # Removed

print(f"Found {len(drone_files)} drone audio files.")
print(f"Found {len(no_drone_files)} non-drone audio files.")
# print(f"Found {len(silence_files)} silence audio files.") # Removed

# Load and display a sample drone audio file and its Mel-spectrogram
if drone_files:
    sample_drone_file = drone_files[0]
    print(f"\nLoading sample drone file: {sample_drone_file}")
    y, sr = librosa.load(sample_drone_file, sr=None) # Load with original sampling rate

    # Display waveform
    plt.figure(figsize=(14, 5))
    librosa.display.waveshow(y, sr=sr)
    plt.title('Sample Drone Audio Waveform')
    plt.show()

    # Extract Mel-spectrogram
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)

    # Display Mel-spectrogram
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Mel-spectrogram of Sample Drone Audio')
    plt.tight_layout()
    plt.show()

else:
    print("No drone audio files found to sample.")

In [ ]:
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Function to extract Mel-spectrogram features
def extract_features(file_path, n_mels=128):
    y, sr = librosa.load(file_path, sr=None) # Load with original sampling rate
    # Using a fixed-size window and hop length for consistent feature dimensions
    # For simplicity, let's target a fixed number of frames per audio file.
    # You might need to adjust this based on the length of your audio files.
    # For this example, let's just use the entire signal and pad/truncate later if needed for a fixed input size model.
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    S_dB = librosa.power_to_db(S, ref=np.max)
    return S_dB

# Prepare data for feature extraction
all_files = drone_files + no_drone_files
all_labels = ["drone"] * len(drone_files) + ["no_drone"] * len(no_drone_files)

# Extract features and labels
X = []
y = []

print("Extracting features...")
for i, file_path in enumerate(all_files):
    try:
        features = extract_features(file_path)
        # Pad or truncate features to a consistent size (e.g., max length found or a common length)
        # For now, let's store them as is and handle padding/truncating during batching or later.
        X.append(features)
        y.append(all_labels[i])
        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(all_files)} files")
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

print("Feature extraction complete.")

# Convert labels to numerical format
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Convert labels to one-hot encoding
y_categorical = to_categorical(y_encoded)

# Find the maximum shape across all extracted features to determine padding/truncating size
max_shape_0 = max([s.shape[0] for s in X])
max_shape_1 = max([s.shape[1] for s in X])

# Pad features to a consistent size (max_shape_0, max_shape_1)
X_padded = np.zeros((len(X), max_shape_0, max_shape_1))
for i, features in enumerate(X):
    s0, s1 = features.shape
    X_padded[i, :s0, :s1] = features


# Split data into training, validation, and test sets
X_train, X_test, y_train, y_test = train_test_split(X_padded, y_categorical, test_size=0.2, random_state=42, stratify=y_categorical)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train) # 0.25 * 0.8 = 0.2

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

The code above performs the following data preprocessing steps:

1.  **`extract_features(file_path, n_mels=128)` function**: This function takes an audio file path and returns its Mel-spectrogram. Mel-spectrograms are a common feature representation for audio classification tasks.
2.  **Gathering Files and Labels**: It combines all drone and non-drone audio file paths and assigns corresponding labels ("drone" or "no_drone").
3.  **Feature Extraction Loop**: It iterates through all audio files, extracts their Mel-spectrogram features using the `extract_features` function, and stores them in the `X` list. It also stores the corresponding labels in the `y` list.
4.  **Label Encoding and One-Hot Encoding**: The categorical labels ("drone", "no_drone") are first converted into numerical labels using `LabelEncoder` and then transformed into a one-hot encoded format using `to_categorical`. This is necessary for training most deep learning models.
5.  **Padding Features**: Since Mel-spectrograms can have varying lengths depending on the audio file duration, the code finds the maximum dimensions and pads all features to a consistent size. This is crucial for creating uniform input for a neural network.
6.  **Data Splitting**: The preprocessed data (`X_padded` and `y_categorical`) is split into three sets: training (60%), validation (20%), and test (20%) using `train_test_split` from `sklearn.model_selection`. The `stratify` parameter ensures that the proportion of classes is maintained in each split.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Reshape data for CNN input (add channel dimension)
X_train_reshaped = X_train[..., np.newaxis]
X_val_reshaped = X_val[..., np.newaxis]
X_test_reshaped = X_test[..., np.newaxis]

input_shape = X_train_reshaped.shape[1:]
num_classes = y_train.shape[1]

# Define the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

The model has been built and compiled. Next, we will train it using the prepared training and validation data.

In [ ]:
epochs = 20  # You can adjust the number of epochs
batch_size = 32 # You can adjust the batch size

history = model.fit(X_train_reshaped, y_train,
                    validation_data=(X_val_reshaped, y_val),
                    epochs=epochs,
                    batch_size=batch_size)

The model training is complete. Now, let's evaluate its performance on the test set to check the accuracy.

In [ ]:
loss, accuracy = model.evaluate(X_test_reshaped, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

# Predict probabilities on the test set
y_pred_proba = model.predict(X_test_reshaped)

# For binary classification, we need the probability of the positive class (drone)
# Assuming 'drone' is the second class (index 1) based on your label encoding.
# You can verify this using label_encoder.classes_ if needed.

# Convert one-hot encoded y_test back to single labels for roc_curve
y_test_labels = np.argmax(y_test, axis=1)

# Calculate ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test_labels, y_pred_proba[:, 1])
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

print(f"AUC Score: {roc_auc:.4f}")

The code above calculates and plots the ROC (Receiver Operating Characteristic) curve and the AUC (Area Under the Curve) score.

*   **ROC Curve**: This plot illustrates the diagnostic ability of a binary classifier system as its discrimination threshold is varied. It plots the True Positive Rate (sensitivity) against the False Positive Rate (1 - specificity) at various threshold settings.
*   **AUC Score**: The AUC represents the degree or measure of separability. It tells us how much the model is capable of distinguishing between classes. The higher the AUC, the better the model is at predicting 0s as 0s and 1s as 1s. An AUC of 1.0 means perfect classification, while an AUC of 0.5 means the model is no better than random guessing.

In [ ]:
import tensorflow as tf

# Convert the Keras model to a TensorFlow Lite model
# Quantization helps reduce model size and computational complexity for TinyML
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
# Ensure that the model uses float32 inputs/outputs even after quantization
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter.representative_dataset = lambda: (
    [X_train_reshaped[i:i+1].astype(np.float32)] for i in range(100) # Use a small subset of training data
)
converter.inference_input_type = tf.float32  # Input type for inference
converter.inference_output_type = tf.float32 # Output type for inference

tflite_model = converter.convert()

# Save the TensorFlow Lite model as a C byte array in a .h file
with open('model.h', 'w') as f:
    f.write('const unsigned char model_tflite[] = {')
    for i, byte in enumerate(tflite_model):
        f.write(f'{byte:#04x}')
        if i != len(tflite_model) - 1:
            f.write(', ')
        if (i + 1) % 12 == 0: # Newline for readability
            f.write('\n ')
    f.write('};')

print("TensorFlow Lite model saved as model.h")
print(f"Model size: {len(tflite_model) / 1024:.2f} KB")

This code performs the following steps to convert your trained Keras model into a TinyML compatible format:

1.  **TensorFlow Lite Conversion**: It uses `tf.lite.TFLiteConverter.from_keras_model(model)` to create a converter object from your trained Keras model.
2.  **Quantization**: `converter.optimizations = [tf.lite.Optimize.DEFAULT]` applies default optimizations, which typically include quantization. Quantization reduces the precision of the model's weights and activations (e.g., from 32-bit floating-point to 8-bit integers), significantly decreasing the model size and improving inference speed on microcontrollers.
3.  **Representative Dataset**: A `representative_dataset` is provided for post-training integer quantization. This dataset helps the converter determine appropriate quantization ranges for activations.
4.  **Target Specification**: It ensures that the model uses float32 for input and output during inference, which is common for many applications.
5.  **Save as C Array (`.h` file)**: The converted TensorFlow Lite model (`tflite_model`) is then written byte by byte into a C header file named `model.h`. This format allows the model to be directly embedded into C/C++ firmware for microcontrollers.